# Experimento: VCP Breakout Volume Threshold

Varia `volume_ratio_threshold` en breakout_params con valores `[1.0, 1.5, 2.0]`
y ejecuta el pipeline completo de deteccion VCP sobre los 23 tickers disponibles.

Cada configuracion se registra en **MLflow** con:
- Graficos de patrones detectados (visualizacion)
- Tabla de simulacion de trades (risk management)
- Metricas: winrate y retorno acumulado por ticker

In [1]:
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import (
    group_signals_into_patterns,
    simulate_trade,
    plot_vcp_pattern,
    plot_trade_simulation,
)

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

Imports OK


## 1. Configuracion

Parametros fijos del pipeline + grilla de valores para `volume_ratio_threshold`.

In [2]:
# --- Grilla de experimento ---
VOLUME_RATIO_THRESHOLDS = [1.0, 1.5, 2.0]

# --- Parametros fijos ---
SWING_CONFIG = ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False)

SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    "max_depth_pct": 0.35,
    "min_total_reduction": 0.80,
    "max_gap_between_contractions_days": None,
}

COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,
}

VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,
}

RISK_PARAMS = {
    "max_stop_loss_pct": 0.07,
    "breakeven_r_multiple": 2.0,
    "trailing_sma_period": 20,
    "trailing_volume_factor": 1.5,
}

# --- Tickers ---
DATA_DIR = project_root / "data" / "csv"
TICKERS = sorted([p.stem for p in DATA_DIR.glob("*.csv")])
print(f"Tickers ({len(TICKERS)}): {TICKERS}")
print(f"Grilla: volume_ratio_threshold = {VOLUME_RATIO_THRESHOLDS}")
print(f"Total corridas: {len(VOLUME_RATIO_THRESHOLDS)} configs x {len(TICKERS)} tickers = {len(VOLUME_RATIO_THRESHOLDS) * len(TICKERS)}")

Tickers (23): ['AAPL', 'AMZN', 'AVGO', 'BRK.B', 'COIN', 'GLD', 'GOOGL', 'HOOD', 'IWM', 'JPM', 'META', 'MSFT', 'NVDA', 'PLTR', 'QLD', 'QQQ', 'SLV', 'SOFI', 'SPY', 'SQQQ', 'TIL', 'TLT', 'TQQQ']
Grilla: volume_ratio_threshold = [1.0, 1.5, 2.0]
Total corridas: 3 configs x 23 tickers = 69


## 2. Funciones auxiliares

In [3]:
def load_ohlc(ticker: str) -> pd.DataFrame:
    path = DATA_DIR / f"{ticker}.csv"
    return pd.read_csv(path, parse_dates=["date"], index_col="date")


def run_ticker_analysis(
    ticker: str,
    ohlc: pd.DataFrame,
    breakout_params: dict,
    risk_params: dict,
) -> dict:
    """Pipeline completo + simulacion de trades para un ticker."""
    swing_detector = ATRZigZagDetector(SWING_CONFIG)

    results = run_full_vcp_pipeline(
        ohlc=ohlc,
        swing_detector=swing_detector,
        sequence_params=SEQUENCE_PARAMS,
        compression_params=COMPRESSION_PARAMS,
        breakout_params=breakout_params,
        volume_contraction_params=VOLUME_CONTRACTION_PARAMS,
    )

    all_signals = {dt: sig for dt, sig in results.items() if sig is not None}
    patterns = group_signals_into_patterns(all_signals, risk_params=risk_params)

    trades = []
    for p in patterns:
        trade = simulate_trade(ohlc, p, risk_params)
        trade["pattern"] = p
        trades.append(trade)

    n_trades = len(trades)
    if n_trades > 0:
        wins = sum(1 for t in trades if t["pnl_pct"] > 0)
        winrate = wins / n_trades
        cumulative_return = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1)
        avg_r_multiple = float(np.mean([t["r_multiple"] for t in trades]))
    else:
        winrate = 0.0
        cumulative_return = 0.0
        avg_r_multiple = 0.0

    return {
        "signals": all_signals,
        "patterns": patterns,
        "trades": trades,
        "metrics": {
            "n_signals": len(all_signals),
            "n_patterns": len(patterns),
            "n_trades": n_trades,
            "winrate": winrate,
            "cumulative_return": cumulative_return,
            "avg_r_multiple": avg_r_multiple,
        },
    }


def build_trade_table(trades: list[dict], ticker: str) -> pd.DataFrame:
    rows = []
    for i, t in enumerate(trades, 1):
        rows.append({
            "ticker": ticker,
            "trade_num": i,
            "entry_date": t["pattern"]["first_signal_date"].strftime("%Y-%m-%d"),
            "exit_date": t["exit_date"].strftime("%Y-%m-%d"),
            "exit_reason": t["exit_reason"],
            "duration_days": t["duration_days"],
            "entry_price": t["pattern"]["entry_price"],
            "exit_price": t["exit_price"],
            "pnl_pct": t["pnl_pct"],
            "r_multiple": t["r_multiple"],
            "max_r": t["max_r"],
            "stop_method": t["pattern"]["stop_method"],
            "n_contractions": t["pattern"]["n_contractions"],
            "atr_ratio": t["pattern"]["atr_ratio"],
        })
    return pd.DataFrame(rows)


print("Funciones auxiliares definidas.")

Funciones auxiliares definidas.


## 3. Setup MLflow

In [4]:
EXPERIMENT_NAME = "VCP_Breakout_Volume_Threshold"
mlflow.set_tracking_uri(str(project_root / "mlruns"))
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow experiment: VCP_Breakout_Volume_Threshold
Tracking URI: /home/gdelarosa/proyectos/deteccion-vcp/mlruns


/home/gdelarosa/proyectos/deteccion-vcp/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## 4. Ejecucion del experimento

Loop principal: por cada valor de `volume_ratio_threshold`, se crea un **parent run** en MLflow.
Dentro, por cada ticker se crea un **child run** con metricas, plots y tablas.

In [5]:
all_summaries = []

ALL_PARAMS = {
    "atr_length": SWING_CONFIG.atr_length,
    "atr_mult": SWING_CONFIG.atr_mult,
    "use_close_only": SWING_CONFIG.use_close_only,
    "seq_method": SEQUENCE_PARAMS["method"],
    "min_contractions": SEQUENCE_PARAMS["min_contractions"],
    "max_contractions": SEQUENCE_PARAMS["max_contractions"],
    "lookback_bars": SEQUENCE_PARAMS["lookback_bars"],
    "tolerance": SEQUENCE_PARAMS["tolerance"],
    "max_depth_pct": SEQUENCE_PARAMS["max_depth_pct"],
    "min_total_reduction": SEQUENCE_PARAMS["min_total_reduction"],
    "compression_method": COMPRESSION_PARAMS["method"],
    "compression_threshold": COMPRESSION_PARAMS["ratio_threshold"],
    "vol_contraction_method": VOLUME_CONTRACTION_PARAMS["method"],
    "vol_contraction_threshold": VOLUME_CONTRACTION_PARAMS["ratio_threshold"],
    "max_stop_loss_pct": RISK_PARAMS["max_stop_loss_pct"],
    "breakeven_r_multiple": RISK_PARAMS["breakeven_r_multiple"],
    "trailing_sma_period": RISK_PARAMS["trailing_sma_period"],
    "trailing_volume_factor": RISK_PARAMS["trailing_volume_factor"],
}

for threshold in VOLUME_RATIO_THRESHOLDS:
    breakout_params = {
        "volume_method": "ratio",
        "volume_ratio_threshold": threshold,
        "volume_lookback_days": 50,
        "require_volume_confirmation": True,
    }

    config_name = f"threshold_{threshold}"
    print(f"\n{'='*70}")
    print(f"CONFIG: {config_name} (volume_ratio_threshold={threshold})")
    print(f"{'='*70}")

    with mlflow.start_run(run_name=config_name) as parent_run:
        mlflow.log_params({
            **ALL_PARAMS,
            "volume_ratio_threshold": threshold,
            "volume_lookback_days": breakout_params["volume_lookback_days"],
        })

        ticker_summaries = []
        all_trades_for_config = []

        for ticker in TICKERS:
            print(f"  {ticker}...", end=" ")
            ohlc = load_ohlc(ticker)

            with mlflow.start_run(run_name=ticker, nested=True) as child_run:
                mlflow.log_params({
                    **ALL_PARAMS,
                    "ticker": ticker,
                    "volume_ratio_threshold": threshold,
                    "volume_lookback_days": breakout_params["volume_lookback_days"],
                    "n_bars": len(ohlc),
                })

                analysis = run_ticker_analysis(ticker, ohlc, breakout_params, RISK_PARAMS)
                metrics = analysis["metrics"]

                mlflow.log_metrics({
                    "n_signals": metrics["n_signals"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                })

                with tempfile.TemporaryDirectory() as tmpdir:
                    for j, pattern in enumerate(analysis["patterns"], 1):
                        plot_path = Path(tmpdir) / f"pattern_{j}.png"
                        plot_vcp_pattern(
                            ohlc, pattern, pattern_number=j,
                            ticker=ticker, save_path=str(plot_path),
                        )
                        mlflow.log_artifact(str(plot_path), "pattern_plots")

                    for j, (pat, trade) in enumerate(
                        zip(analysis["patterns"], analysis["trades"]), 1
                    ):
                        trade_plot_path = Path(tmpdir) / f"trade_{j}.png"
                        plot_trade_simulation(
                            ohlc, pat, trade, pattern_number=j,
                            risk_params=RISK_PARAMS, ticker=ticker,
                            save_path=str(trade_plot_path),
                        )
                        mlflow.log_artifact(str(trade_plot_path), "trade_plots")

                    if analysis["trades"]:
                        trade_df = build_trade_table(analysis["trades"], ticker)
                        trade_csv_path = Path(tmpdir) / "trades.csv"
                        trade_df.to_csv(trade_csv_path, index=False)
                        mlflow.log_artifact(str(trade_csv_path), "tables")
                        all_trades_for_config.append(trade_df)

                ticker_summaries.append({
                    "ticker": ticker,
                    "n_signals": metrics["n_signals"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                })

                print(
                    f"{metrics['n_patterns']} pat, "
                    f"{metrics['n_trades']} trades, "
                    f"WR={metrics['winrate']:.0%}, "
                    f"CR={metrics['cumulative_return']:+.1%}"
                )

        summary_df = pd.DataFrame(ticker_summaries)
        summary_df["volume_ratio_threshold"] = threshold
        all_summaries.append(summary_df)

        tickers_with_trades = summary_df[summary_df["n_trades"] > 0]

        agg_metrics = {
            "total_patterns": int(summary_df["n_patterns"].sum()),
            "total_trades": int(summary_df["n_trades"].sum()),
            "tickers_with_patterns": int((summary_df["n_patterns"] > 0).sum()),
            "tickers_with_trades": int((summary_df["n_trades"] > 0).sum()),
            "avg_winrate": float(tickers_with_trades["winrate"].mean())
            if len(tickers_with_trades) > 0
            else 0.0,
            "median_winrate": float(tickers_with_trades["winrate"].median())
            if len(tickers_with_trades) > 0
            else 0.0,
            "avg_cumulative_return": float(tickers_with_trades["cumulative_return"].mean())
            if len(tickers_with_trades) > 0
            else 0.0,
            "avg_r_multiple": float(tickers_with_trades["avg_r_multiple"].mean())
            if len(tickers_with_trades) > 0
            else 0.0,
        }
        mlflow.log_metrics(agg_metrics)

        with tempfile.TemporaryDirectory() as tmpdir:
            summary_path = Path(tmpdir) / f"summary_{config_name}.csv"
            summary_df.to_csv(summary_path, index=False)
            mlflow.log_artifact(str(summary_path), "summaries")

            if all_trades_for_config:
                all_trades_df = pd.concat(all_trades_for_config, ignore_index=True)
                all_trades_path = Path(tmpdir) / f"all_trades_{config_name}.csv"
                all_trades_df.to_csv(all_trades_path, index=False)
                mlflow.log_artifact(str(all_trades_path), "summaries")

        print(
            f"\n  AGREGADO: {agg_metrics['total_patterns']} patrones, "
            f"{agg_metrics['total_trades']} trades, "
            f"avg WR={agg_metrics['avg_winrate']:.0%}, "
            f"avg CR={agg_metrics['avg_cumulative_return']:+.1%}"
        )

print("\nExperimento completo!")


CONFIG: threshold_1.0 (volume_ratio_threshold=1.0)


  AAPL... 

/home/gdelarosa/proyectos/deteccion-vcp/vcp_detection/analysis.py:363: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


/home/gdelarosa/proyectos/deteccion-vcp/vcp_detection/analysis.py:540: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


7 pat, 7 trades, WR=86%, CR=+71.6%
  AMZN... 

...

1 pat, 1 trades, WR=0%, CR=-3.8%
  TLT... 

/home/gdelarosa/proyectos/deteccion-vcp/vcp_detection/analysis.py:363: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


/home/gdelarosa/proyectos/deteccion-vcp/vcp_detection/analysis.py:540: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


1 pat, 1 trades, WR=0%, CR=-3.0%
  TQQQ... 

0 pat, 0 trades, WR=0%, CR=+0.0%

  AGREGADO: 38 patrones, 38 trades, avg WR=42%, avg CR=+2.8%

Experimento completo!


## 5. Comparacion entre configuraciones

Tablas pivoteadas: winrate y retorno acumulado por ticker para cada valor de threshold.

In [6]:
comparison_df = pd.concat(all_summaries, ignore_index=True)

pivot_winrate = comparison_df.pivot(
    index="ticker", columns="volume_ratio_threshold", values="winrate"
)
pivot_cumret = comparison_df.pivot(
    index="ticker", columns="volume_ratio_threshold", values="cumulative_return"
)
pivot_patterns = comparison_df.pivot(
    index="ticker", columns="volume_ratio_threshold", values="n_patterns"
)

print("=== Win Rate por Ticker y Threshold ===")
display(pivot_winrate.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

print("\n=== Retorno Acumulado por Ticker y Threshold ===")

def color_returns(val):
    if val > 0:
        return "background-color: #27ae60; color: white"
    elif val < 0:
        return "background-color: #e74c3c; color: white"
    return ""

display(pivot_cumret.style.format("{:+.1%}").applymap(color_returns))

print("\n=== Patrones Detectados por Ticker y Threshold ===")
display(pivot_patterns.style.format("{:.0f}").background_gradient(cmap="Blues"))

=== Win Rate por Ticker y Threshold ===


volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,86%,83%,100%
AMZN,29%,29%,29%
AVGO,71%,67%,33%
BRK.B,17%,33%,0%
COIN,0%,0%,0%
GLD,50%,50%,100%
GOOGL,64%,60%,100%
HOOD,0%,0%,0%
IWM,14%,0%,0%



=== Retorno Acumulado por Ticker y Threshold ===


/tmp/ipykernel_353307/1506673597.py:25: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(pivot_cumret.style.format("{:+.1%}").applymap(color_returns))


volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,+71.6%,+26.5%,+10.6%
AMZN,-12.0%,-28.7%,-26.8%
AVGO,+35.6%,-2.5%,-12.8%
BRK.B,-14.5%,-4.2%,+0.0%
COIN,-34.1%,-18.2%,-10.9%
GLD,+11.8%,+8.1%,+37.6%
GOOGL,+82.7%,+47.3%,+49.6%
HOOD,-18.4%,-10.2%,-10.2%
IWM,-19.0%,-8.9%,-6.9%



=== Patrones Detectados por Ticker y Threshold ===


volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,7,6,3
AMZN,17,14,7
AVGO,7,6,3
BRK.B,6,3,0
COIN,3,2,1
GLD,4,4,2
GOOGL,11,10,3
HOOD,2,1,1
IWM,7,2,1


## 6. Grafico comparativo agregado

In [7]:
matplotlib.use("Agg")

# Filtrar tickers sin trades para que el promedio coincida con las metricas de MLflow
traded_df = comparison_df[comparison_df["n_trades"] > 0]

agg_by_threshold = traded_df.groupby("volume_ratio_threshold").agg({
    "n_patterns": "sum",
    "n_trades": "sum",
    "winrate": "mean",
    "cumulative_return": "mean",
}).reset_index()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col, title, fmt in zip(
    axes,
    ["n_patterns", "n_trades", "winrate", "cumulative_return"],
    ["Total Patrones", "Total Trades", "Avg Win Rate", "Avg Retorno Acum."],
    ["{:.0f}", "{:.0f}", "{:.0%}", "{:+.1%}"],
):
    bars = ax.bar(
        agg_by_threshold["volume_ratio_threshold"].astype(str),
        agg_by_threshold[col],
        color=["#3498db", "#e67e22", "#27ae60"],
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("volume_ratio_threshold")
    for bar, val in zip(bars, agg_by_threshold[col]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            fmt.format(val),
            ha="center",
            va="bottom",
            fontsize=10,
        )

plt.tight_layout()
chart_path = project_root / "experiments" / "threshold_comparison.png"
fig.savefig(str(chart_path), dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Grafico guardado en: {chart_path}")

display(agg_by_threshold.style.format({
    "n_patterns": "{:.0f}",
    "n_trades": "{:.0f}",
    "winrate": "{:.0%}",
    "cumulative_return": "{:+.1%}",
}))

Grafico guardado en: /home/gdelarosa/proyectos/deteccion-vcp/experiments/threshold_comparison.png


,volume_ratio_threshold,n_patterns,n_trades,winrate,cumulative_return
0,1.000000,127,127,46%,+20.9%
1,1.500000,86,86,41%,+5.6%
2,2.000000,38,38,42%,+2.8%


## 7. Conclusiones

Para explorar los resultados en detalle:
```bash
cd /home/gdelarosa/proyectos/deteccion-vcp && mlflow ui --backend-store-uri mlruns/
```

Cada parent run contiene:
- **summaries/**: CSV con metricas por ticker y CSV con todos los trades
- Metricas agregadas (avg winrate, avg cumulative return)

Cada child run contiene:
- **pattern_plots/**: Graficos de cada patron VCP detectado
- **trade_plots/**: Graficos de simulacion de cada trade
- **tables/**: CSV con detalle de trades